# ENG-DRB OpenAI reproduction

This notebook reproduces the **OpenAI backend** of the ENG-DRB benchmark.

It prepares request JSONL files locally, then uses the OpenAI **Batch API** for asynchronous execution.

In [ ]:
%pip install -q -U openai datasets tqdm
import os, sys, json
from pathlib import Path
if not Path('src').exists():
    !git clone YOUR_GITHUB_REPO_URL repo
    %cd repo
sys.path.insert(0, str(Path('src').resolve()))

In [ ]:
from eng_drb_benchmark.data import load_eng_drb, export_gold_jsonl
from eng_drb_benchmark.batch import create_openai_batch_requests
from eng_drb_benchmark.postprocess import merge_openai_batch_results, deduplicate_prediction_file
from eng_drb_benchmark.evaluate import evaluate_from_files

RELATION_TYPE = 'non_implicit'   # or 'implicit'
PROMPT_FILE = f'prompts/{RELATION_TYPE}.txt'
OUTPUT_DIR = Path(f'results/openai_{RELATION_TYPE}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL = 'o4-mini-2025-04-16'
WINDOW_SIZE = 20
STEP = 10

In [ ]:
dataset_dict = load_eng_drb()
split_name = next(iter(dataset_dict.keys()))
dataset = dataset_dict[split_name]
gold_path = export_gold_jsonl(dataset, OUTPUT_DIR / f'gold_{RELATION_TYPE}.jsonl', relation_type=RELATION_TYPE)
prompt_text = Path(PROMPT_FILE).read_text(encoding='utf-8')
request_path = create_openai_batch_requests(
    records=(json.loads(line) for line in gold_path.read_text(encoding='utf-8').splitlines() if line.strip()),
    output_path=OUTPUT_DIR / f'requests_{RELATION_TYPE}.jsonl',
    prompt_text=prompt_text,
    model=MODEL,
    window_size=WINDOW_SIZE,
    step=STEP,
)
print(request_path)

Submit and download the Batch job either from the terminal scripts in the README or by adding equivalent OpenAI client cells here.
Once `batch_results.jsonl` exists, run the next cell.

In [ ]:
batch_results = OUTPUT_DIR / 'batch_results.jsonl'
merged_path = merge_openai_batch_results(batch_results, OUTPUT_DIR / 'pred_merged.jsonl')
dedup_path = deduplicate_prediction_file(merged_path, OUTPUT_DIR / 'pred_dedup.jsonl')
scores = evaluate_from_files(gold_path, dedup_path)
print(json.dumps(scores['partial_agreement']['overall_scores'], indent=2))
print(json.dumps(scores['exact_match']['overall_scores'], indent=2))